# Milestone 2: frozen-model size intervention

Evaluate the frozen **5,760-update** model with original scenes and four circle/square size combinations: (6,6), (6,8), (8,6), (8,8). Keep object origins, identities, colors, triangle size and questions fixed. No training or test inference.

Enable GPU and internet, attach `milestone2_training_budget_artifacts.zip` (or its extracted directory), and set `REFERENCE_SOURCE`. Push the implementation and set a committed `REPO_REF` before running. One GPU (`cuda:0`), float32, batch 32, R=2; preserve Kaggle's PyTorch installation.

Use the same **384 eligible validation images** in every condition: **5,760 QA evaluations** including originals. Objects must remain fully visible and separated; edge contact is allowed and vertical center alignment is relaxed only in this diagnostic. Eligibility is decided before predictions.

See `docs/milestones/milestone2_size_intervention.md`. This experiment is descriptive: no new pass/fail gates, no parameter changes, and no milestone-completion claim. Use fresh output paths.


In [ ]:
REPO_URL = "https://github.com/Krailon/multi-modal-loop-llm.git"
REPO_REF = "milestone2"  # A committed revision containing this notebook; commit SHA preferred.
REPO_DIR = "/kaggle/working/multi-modal-loop-size-intervention"
RUN_ROOT = "/kaggle/working/milestone2_size_intervention"
REFERENCE_SOURCE = (
    "/kaggle/input/REPLACE_WITH_YOUR_ARTIFACT_PATH/milestone2_training_budget_artifacts.zip"
)

## Checkout and install

Run all cells in order. The resolved revision is recorded with the artifacts.


In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR).resolve()
RUN_ROOT = Path(RUN_ROOT).resolve()
if REPO_DIR == RUN_ROOT or REPO_DIR.is_relative_to(RUN_ROOT) or RUN_ROOT.is_relative_to(REPO_DIR):
    raise ValueError("Repository and artifacts must use separate directories")
if not REPO_REF or REPO_REF.startswith("-"):
    raise ValueError("Set REPO_REF to a branch, tag, or commit")


def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()


if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
else:
    if git("remote", "get-url", "origin") != REPO_URL:
        raise ValueError("Existing checkout belongs to a different repository")
    if git("status", "--porcelain"):
        raise ValueError("Existing checkout has local changes; use a clean committed checkout")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    if git("rev-parse", "HEAD") != git("rev-parse", "FETCH_HEAD"):
        raise ValueError(
            "Existing checkout has a different revision. Set REPO_REF to its recorded commit "
            "to reuse this checkout, or use a new REPO_DIR for a new run."
        )

resolved_revision = git("rev-parse", "HEAD")
identity = (str(REPO_DIR), resolved_revision)
if globals().get("_notebook_code_identity", identity) != identity:
    raise RuntimeError(
        "The kernel previously loaded another revision; restart it before proceeding"
    )
_notebook_code_identity = identity
print("Resolved code revision:", resolved_revision)
torch_version = importlib.metadata.version("torch")
# No torch extra, requirements.txt, or accelerator replacement.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)
if importlib.metadata.version("torch") != torch_version:
    raise RuntimeError("PyTorch changed during installation; inspect the environment")
# Editable-install .pth files are processed at interpreter startup. Make this
# checkout importable immediately in the running notebook kernel too.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
# Training/evaluation run in subprocesses using this kernel's Python interpreter.

## Audit the reference and record the protocol

Check the completed training-budget checkpoint, settings, manifest and reports against their recorded identities. Build eligibility and all five conditions locally before inference. Excluded source scenes and reasons are retained.


In [ ]:
import multimodal_loop.eval.kaggle_size_intervention as helpers
from multimodal_loop.eval.kaggle_size_intervention import (
    archive_size_intervention,
    prepare_size_intervention,
    run_size_intervention,
)

if not Path(helpers.__file__).resolve().is_relative_to(REPO_DIR / "src"):
    raise RuntimeError("Another package checkout is cached; restart the kernel")
run = prepare_size_intervention(REPO_DIR, RUN_ROOT, REFERENCE_SOURCE)

## Frozen evaluation

Evaluate only the original and matched size variants on validation. Both circle/square answers and the unchanged triangle answer are measured. Record any disagreement between original-scene predictions and their saved reference predictions. The checkpoint hash must remain unchanged.


In [ ]:
report = run_size_intervention(run)

## Inspect descriptive results

Compare per-condition/per-shape/per-layout accuracy, pair accuracy, size-reversal transitions, correct invariance and stable incorrect answers. Triangle changes diagnose effects on the unchanged object. Resizing changes contour, area, center, patch coverage and sometimes edge contact; it does not isolate an internal mechanism.


In [ ]:
import json

from IPython.display import HTML, FileLink, display

print(json.dumps(report["eligibility"]["eligible_by_geometry"], indent=2))
print(json.dumps(report["metrics"], indent=2))
print("Original prediction agreement:", report["original_reference_agreement"])
display(HTML((run.root / "diagnosis" / "inspection.html").read_text()))


## Retain artifacts

Download `milestone2_size_intervention_artifacts.zip` for review. It includes original/variant metadata, eligibility, predictions, summaries, preview, hashes, runtime/revision provenance and logs. The staged reference checkpoint is excluded. No new checkpoint is trained.


In [ ]:
archive = archive_size_intervention(run)
print("Size-intervention archive:", archive)
display(FileLink(str(archive)))